# 인공지능응용 · Week 13 · 긴 글을 나누어 다음 문자 예측하기

**본인 사본을 만들어 실행·수정·기록하세요.**

- 이름: **⟦여기에 직접 입력⟧**
- 학번: **⟦여기에 직접 입력⟧**

[자습 자료](https://chorok-daddy.github.io/courses/ai-applications/lstm-gru/index.html) · [실습 안내](https://chorok-daddy.github.io/courses/ai-applications/lstm-gru/assignment.html)

## 시작 전 · 셀 실행과 작성 방법
Code Cell 왼쪽 ▶ 또는 Shift+Enter로 실행하세요. 위에서부터 진행하며 앞 셀의 변수를 사용합니다. Text Cell은 더블클릭해 **⟦직접 입력⟧** 부분을 바꾸고 Shift+Enter로 표시합니다. 기준 예제는 실행해서 이해하고, **직접 작성** 셀에 본인 코드를 작성하세요. 값을 바꾸면 해당 셀과 뒤의 관련 셀을 다시 실행합니다.

**60분 진행:** 예상한 결과를 기록하고 실행한 뒤, 조건을 바꾸어 비교하세요. 마지막에는 새 세션에서 다시 실행하고 작성한 설명과 파일을 확인하세요. 시간이 남으면 마지막 선택 실습을 수행하세요.

Colab에 필요한 라이브러리가 없으면 새 런타임에서 환경을 확인하고 조교에게 문의하세요. CPU로 기준 실습을 실행할 수 있습니다. 외부 다운로드가 필요한 실습은 해당 셀에 표시합니다.

## 1. 긴 글에서 짧은 학습 예제 만들기

`hello!`를 `h → e → l → l → o → !`처럼 순서대로 읽어 보세요. 이렇게 순서가 있는 나열이 **Sequence**입니다. 문자 하나가 한 시점이며 공백도 한 문자입니다.

원문에서 일정한 길이로 잘라 보는 구간을 **Window**라고 합니다. 길이 3으로 `hel`을 읽을 때 각 위치의 다음 문자 정답은 `ell`입니다. `h` 다음은 `e`, `he` 다음은 `l`, `hel` 다음은 `l`입니다. 정답은 마지막 하나만이 아니라 각 위치마다 있습니다.

| 예제 | 원문 시작 위치 | Input | Target |
|---|---:|---|---|
| 1 | 0 | `hel` | `ell` |
| 2 | 2 | `llo` | `lo!` |

**Window Size=3**은 입력 길이, **Stride=2**는 다음 예제의 시작점을 옮기는 간격입니다. Target은 언제나 Input보다 **한 글자 뒤**입니다. Stride가 2여도 두 글자 뒤를 예측하지 않습니다. 예제 1과 2는 원문 위치 2를 함께 포함합니다.

두 예제를 행으로 쌓으면 Batch가 됩니다. 행 수 N=2, 행 안의 문자 수 S=3이며, 문자마다 길이 E의 One-hot 벡터를 넣으면 Input shape는 `(2,3,E)`입니다. 한 행 안에서 State는 다음 문자로 전달되지만, 다른 행으로 넘기지는 않습니다.

### A-1. 짧은 문자열로 자르는 규칙 확인하기


In [ ]:
# 한 예제의 입력 길이와 다음 예제로 이동하는 간격을 구별하세요.
demo_text = "hello!"
demo_size, demo_stride = 3, 2
for start in range(0, len(demo_text) - demo_size, demo_stride):
    inp = demo_text[start:start + demo_size]
    target = demo_text[start + 1:start + demo_size + 1]
    print(f"시작 위치 {start}: Input={inp!r}, Target={target!r}")


### A-2. 긴 글에 같은 규칙 적용하기

이제 입력 길이를 20, 이동 간격을 5로 정합니다. 시작 위치는 0·5·10·…이며, 각 Target은 Input보다 한 글자 뒤에서 시작합니다. 마지막 정답까지 있으려면 입력 길이보다 한 글자 더 필요합니다. 만든 예제를 모두 한 Batch로 묶으며, 실제 N·S·E는 출력으로 확인하세요.


In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(42)
torch.set_num_threads(2)
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
sentences=["if you want to build a ship, don't drum up people together to", "collect wood and don't assign them tasks and work, but rather", "teach them to long for the endless immensity of the sea."]
sentence=' '.join(sentences);chars=sorted(set(sentence));vocab={c:i for i,c in enumerate(chars)}
window_size = 20
window_stride = 5
starts = range(0, len(sentence) - window_size, window_stride)
ids = torch.tensor([[vocab[c] for c in sentence[i:i+window_size]] for i in starts])
Y = torch.tensor([[vocab[c] for c in sentence[i+1:i+window_size+1]] for i in starts])
X = F.one_hot(ids, num_classes=len(chars)).float()
print('X:', X.shape, 'Y:', Y.shape)
for i in range(2):
    print('input:', ''.join(chars[j] for j in ids[i]))
    print('target:', ''.join(chars[j] for j in Y[i]))
assert X.shape[:2] == Y.shape


### B. 직접 수행

`hello!`를 길이 3·Stride 2로 잘라 만든 두 Input·Target 쌍을 확인하세요. 긴 글의 첫 두 쌍도 출력하고, 각 쌍의 한 글자 차이와 예제 시작점의 이동 간격을 구별하세요. X·Y의 shape에서 Window 수 N, 길이 S, 문자 표현 크기 E를 적으세요. State 반환값은 다음 Section에서 모델을 만든 뒤 확인합니다.

예상은 정확한 수치 대신 shape나 증가·감소 방향으로 적어도 됩니다. 설명은 아래 Text Cell에 기록하세요.


In [ ]:
# ✍ 직접 작성: 이 셀 아래에 본인의 코드를 추가하세요.
# 코드가 길어지면 Code Cell을 추가해도 됩니다.


### C. 직접 기록 · 이 Text Cell을 더블클릭해서 수정

- 확인하거나 바꾼 조건: **⟦직접 입력⟧**
- 실행 전 예상: **⟦직접 입력⟧**
- 실제 출력/그래프에서 확인한 값: **⟦직접 입력⟧**
- 예상과 차이 및 설명: **⟦직접 입력⟧**

## 2. RNN·LSTM·GRU 비교

같은 Window 데이터와 학습 조건으로 RNN·LSTM·GRU를 비교합니다. 각 모델은 Hidden Size=32이며, 모든 시점의 output을 Linear에 넣어 문자 점수를 만듭니다. LSTM만 Hidden State 외에 Cell State를 반환합니다. 학습 결과는 results에 저장되므로 표와 State 확인을 위해 다시 학습할 필요는 없습니다.

### A. 기준 예제 · 먼저 읽고 실행


In [ ]:
class CharModel(nn.Module):
    def __init__(self,kind,vocab_size,hidden=32):
        super().__init__();self.rnn=getattr(nn,kind)(vocab_size,hidden,batch_first=True);self.fc=nn.Linear(hidden,vocab_size)
    def forward(self,x):
        output,state=self.rnn(x);return self.fc(output)
def fit_char(kind,X,Y,epochs=300,lr=.01):
    torch.manual_seed(42);m=CharModel(kind,X.shape[-1]);opt=torch.optim.Adam(m.parameters(),lr=lr);losses=[]
    for epoch in range(epochs):
        logits=m(X);loss=F.cross_entropy(logits.reshape(-1,X.shape[-1]),Y.reshape(-1))
        opt.zero_grad();loss.backward();opt.step();losses.append(loss.item())
        if epoch%100==0:print(kind,epoch,loss.item())
    with torch.no_grad():pred=m(X).argmax(-1)
    return m,losses,pred
results={}
for kind in ['RNN','LSTM','GRU']:
    m,h,p=fit_char(kind,X,Y,epochs=300,lr=.01);results[kind]=(m,h,p)
    print(kind,'accuracy:',(p==Y).float().mean().item());print(''.join(chars[i] for i in p[0]))
    plt.plot(h,label=kind)
plt.legend();plt.xlabel('Epoch');plt.ylabel('Loss');plt.grid();plt.show()
with torch.no_grad():output,(h_n,c_n)=results['LSTM'][0].rnn(X)
print('LSTM output:',output.shape,'h_n:',h_n.shape,'c_n:',c_n.shape)
for kind in ['RNN', 'GRU']:
    with torch.no_grad(): output, h_n = results[kind][0].rnn(X)
    print(kind, 'output:', output.shape, 'h_n:', h_n.shape)



### B. 직접 수행

RNN·LSTM·GRU의 마지막 Loss·Training Accuracy·첫 Window의 예측을 표로 기록하세요. 같은 데이터·epoch·Learning Rate를 사용했는지 확인합니다. output과 마지막 h_n의 shape를 비교하고 LSTM에만 c_n이 있는 이유를 설명하세요. 세 모델의 결과로 반복 기록표를 채우면 됩니다.

예상은 정확한 수치 대신 shape나 증가·감소 방향으로 적어도 됩니다. 설명은 아래 Text Cell에 기록하세요.


In [ ]:
# ✍ 직접 작성: 이 셀 아래에 본인의 코드를 추가하세요.
# 코드가 길어지면 Code Cell을 추가해도 됩니다.


### C. 직접 기록 · 이 Text Cell을 더블클릭해서 수정

- 확인하거나 바꾼 조건: **⟦직접 입력⟧**
- 실행 전 예상: **⟦직접 입력⟧**
- 실제 출력/그래프에서 확인한 값: **⟦직접 입력⟧**
- 예상과 차이 및 설명: **⟦직접 입력⟧**

## 반복 숙달 · 실행 전에 판단하기

앞서 비교한 RNN·LSTM·GRU의 결과를 이 표에 정리하세요. 같은 모델을 세 번 더 학습할 필요는 없습니다. 같은 데이터·epoch·Learning Rate를 사용했는지 확인합니다.

| 변경 조건 | 예상 shape/수치/결과 | 실제 결과 | 오류가 있었다면 원인 |
|---|---|---|---|
| ⟦직접 입력 1⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |
| ⟦직접 입력 2⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |
| ⟦직접 입력 3⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |

In [ ]:
# ✍ 반복 실험 코드


## 선택 실습 · 오류의 원인을 찾아 코드 고치기

앞 코드의 축·dtype·입력 크기·모델 설정 중 하나를 일부러 바꿔 예상과 달라지는 사례를 만드세요. 오류를 그대로 남기지 말고, 어떤 입력 조건이나 연산 규칙이 맞지 않았는지 설명한 뒤 수정된 코드로 실행하세요. 인증·설치 설정을 바꾸는 실험은 하지 않습니다.

## 저장 전 확인

1. 직접 작성란을 채우고 필요한 출력·그래프를 남겼는지 확인합니다.
2. 새 런타임에서 위에서부터 실행해 숨은 변수 의존성을 확인합니다.
3. 수정한 파일을 본인 Drive에 저장하고 `.ipynb`로 내려받습니다.
4. 제출 파일명·기한은 블로그의 이번 주 실습 안내와 최신 KLAS 공지를 따릅니다.